# 21.1 圖片切成 patch

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/21-patches/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""追蹤 32×32 RGB 圖的 patch 順序、共用投影、位置向量與 CLS。"""

import json

import torch

from miniyolo.vision_transformer import PatchEmbedding, TinyViT, patchify


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    # 示意材料：每個8×8 patch 的 R 是編號/16，G=.25，B=.75；不是訓練資料。
    image = torch.empty(1, 3, 32, 32)
    for index in range(16):
        row, col = divmod(index, 4)
        image[0, :, row * 8:(row + 1) * 8, col * 8:(col + 1) * 8] = torch.tensor(
            [index / 16, .25, .75])[:, None, None]
    raw = patchify(image, 8)
    assert raw.shape == (1, 16, 192)
    assert torch.equal(raw[0, :, 0], torch.arange(16) / 16)
    projection = PatchEmbedding(image_size=32, patch_size=8, embed_dim=3)
    with torch.no_grad():
        projection.projection.weight.zero_()
        projection.projection.bias.zero_()
        for channel in range(3):
            projection.projection.weight[channel, channel] = 1 / 64
    projected = projection(image)
    assert torch.allclose(projected[0, :, 0], torch.arange(16) / 16)
    assert torch.allclose(projected[0, :, 1:], torch.tensor([.25, .75]).expand(16, 2))
    model = TinyViT()
    tokens = model.embed_tokens(image)
    patch_tokens = model.patch_embed(image)
    assert torch.allclose(tokens[:, 0], model.cls_token[:, 0] + model.pos_embed[:, 0])
    assert torch.allclose(tokens[:, 1:], patch_tokens + model.pos_embed[:, 1:])
    # 相同patch內容仍能帶不同位置資訊；pos向量是待訓練參數，並非座標答案。
    uniform = torch.ones_like(image) * .25
    uniform_without_position = model.embed_tokens(uniform, use_position=False)
    uniform_with_position = model.embed_tokens(uniform)
    assert torch.equal(uniform_without_position[:, 1], uniform_without_position[:, 2])
    assert not torch.equal(uniform_with_position[:, 1], uniform_with_position[:, 2])
    result = {
        "event": "patches", "image_shape": list(image.shape), "patch_size": 8,
        "raw_patch_shape": list(raw.shape), "raw_vector_order": "channel,row,column",
        "patch_order": [[i, i // 4, i % 4] for i in range(16)],
        "patch_red_means": projected[0, :, 0].detach().tolist(),
        "projection_shape": list(projected.shape), "projection_demo": "RGB means; hand-set weights",
        "vit_patch_embedding_shape": list(patch_tokens.shape),
        "tokens_with_cls_shape": list(tokens.shape), "cls_index": 0,
        "first_patch_sequence_index": 1, "learned_position_shape": list(model.pos_embed.shape),
        "same_content_equal_without_position": bool(torch.equal(uniform_without_position[:, 1], uniform_without_position[:, 2])),
        "same_content_equal_with_position": bool(torch.equal(uniform_with_position[:, 1], uniform_with_position[:, 2])),
        "full_attention_scores_per_head": {"patch8_with_cls": 17 ** 2, "patch4_with_cls": 65 ** 2},
        "score_count_ratio_with_cls": 65 ** 2 / 17 ** 2,
        "patch_only_score_count_ratio": 64 ** 2 / 16 ** 2,
        "limitation": "Hand-set projection explains token construction; no optimizer update or learned classification here.",
    }
    print(json.dumps(result, ensure_ascii=False, allow_nan=False))
    return result


if __name__ == "__main__":
    main()


{"event": "patches", "image_shape": [1, 3, 32, 32], "patch_size": 8, "raw_patch_shape": [1, 16, 192], "raw_vector_order": "channel,row,column", "patch_order": [[0, 0, 0], [1, 0, 1], [2, 0, 2], [3, 0, 3], [4, 1, 0], [5, 1, 1], [6, 1, 2], [7, 1, 3], [8, 2, 0], [9, 2, 1], [10, 2, 2], [11, 2, 3], [12, 3, 0], [13, 3, 1], [14, 3, 2], [15, 3, 3]], "patch_red_means": [0.0, 0.0625, 0.125, 0.1875, 0.25, 0.3125, 0.375, 0.4375, 0.5, 0.5625, 0.625, 0.6875, 0.75, 0.8125, 0.875, 0.9375], "projection_shape": [1, 16, 3], "projection_demo": "RGB means; hand-set weights", "vit_patch_embedding_shape": [1, 16, 32], "tokens_with_cls_shape": [1, 17, 32], "cls_index": 0, "first_patch_sequence_index": 1, "learned_position_shape": [1, 17, 32], "same_content_equal_without_position": true, "same_content_equal_with_position": false, "full_attention_scores_per_head": {"patch8_with_cls": 289, "patch4_with_cls": 4225}, "score_count_ratio_with_cls": 14.619377162629759, "patch_only_score_count_ratio": 16.0, "limitation